In [1]:
!pip install gradio

In [2]:
import gradio as gr

def chatbot(message, history):
    question = message.lower()

    if "attendance" in question:
        return (
            "Attendance question received! "
            "I'll provide the official SSIT attendance "
            "rules once the college documents are connected."
        )

    elif "hostel" in question:
        return (
            "Hostel question received! "
            "The official hostel rules will be connected "
            "when the documents are available."
        )

    elif "exam" in question:
        return (
            "Exam question received! "
            "I'll use the official examination regulations "
            "once they're connected."
        )

    else:
        return (
            "I'm the SSIT Campus Regulations Assistant. "
            "Please ask about attendance, examinations, "
            "or hostel rules."
        )

demo = gr.ChatInterface(
    fn=chatbot,
    title="SSIT Campus Regulations Assistant",
    description=(
        "Ask questions about SSIT attendance, "
        "examinations, and hostel rules."
    ),
    examples=[
        "What are the attendance rules?",
        "Tell me about examination regulations",
        "What are the hostel rules?"
    ]
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c575302939df19c998.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
!pip install -q gradio sentence-transformers faiss-cpu transformers torch

In [5]:
documents = [
    {
        "text": "Students must follow the attendance "
                "requirements specified in the official "
                "college regulations.",
        "source": "sample_attendance.txt",
        "page": 1
    },
    {
        "text": "Students must follow the hostel's "
                "official visiting hours and entry rules.",
        "source": "sample_hostel.txt",
        "page": 1
    },
    {
        "text": "Students must follow the examination "
                "rules published by the college.",
        "source": "sample_exams.txt",
        "page": 1
    }
]

In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [doc["text"] for doc in documents]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.asarray(embeddings, dtype="float32"))

def retrieve(question, k=2):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    scores, ids = index.search(
        np.asarray(question_embedding, dtype="float32"),
        min(k, len(documents))
    )

    return [
        documents[i]
        for i in ids[0]
        if i >= 0
    ]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

# Using the Auto classes ensures we can load google/flan-t5-small directly
model_name = "google/flan-t5-small"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BailingHybridForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BeamForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCa

In [9]:
def answer_question(question):
    results = retrieve(question)

    context = "\n".join(
        f"Source: {doc['source']}, "
        f"page {doc['page']}\n{doc['text']}"
        for doc in results
    )

    prompt = f"""
Answer the question using only the context below.
If the context does not contain the answer,
say that you could not find the answer
in the available documents.

Context:
{context}

Question: {question}

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=120
    )[0]["generated_text"]

    sources = "\n".join(
        f"- {doc['source']}, page {doc['page']}"
        for doc in results
    )

    return f"{result}\n\n**Sources:**\n{sources}"

In [10]:
import gradio as gr

def chatbot(message, history):
    return answer_question(message)

demo = gr.ChatInterface(
    fn=chatbot,
    title="SSIT Campus Regulations Assistant",
    description=(
        "Ask about attendance, examinations, "
        "and hostel rules."
    ),
    examples=[
        "What are the attendance rules?",
        "What are the hostel rules?",
        "Where can I find examination regulations?"
    ]
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4c70150f808cc5791e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
import gradio as gr
print(gr.__version__)

6.30.0


In [14]:

import gradio as gr

def chatbot(message, history):
    return answer_question(message)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎓 SSIT Campus Assistant
    ### Sri Siddhartha Institute of Technology

    Ask questions about attendance, examinations,
    and hostel regulations.
    """)

    gr.ChatInterface(
        fn=chatbot,
        examples=[
            "What are the attendance rules?",
            "What are the hostel rules?",
            "What are the examination rules?"
        ],
        chatbot=gr.Chatbot(height=420),
        textbox=gr.Textbox(
            placeholder="Type your question here...",
            lines=2
        )
    )

demo.launch(share=True, debug=True)


/tmp/ipykernel_2295/3390166139.py:6: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e2f09d5dde25c8b547.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 

In [15]:

import gradio as gr

custom_css = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
    font-family: Arial, sans-serif !important;
}

#hero {
    background: linear-gradient(135deg, #102A43, #176B87);
    color: white;
    padding: 28px;
    border-radius: 18px;
    margin-bottom: 18px;
}

#hero h1 {
    color: white;
    font-size: 30px;
}

#hero p {
    color: #E6F4F7;
    font-size: 15px;
}

#chat-area {
    border-radius: 16px;
}

#footer {
    text-align: center;
    color: #64748B;
    font-size: 12px;
    padding: 15px;
}
"""

def chatbot(message, history):
    return answer_question(message)

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css
) as demo:

    gr.HTML("""
    <div id="hero">
        <h1>🎓 SSIT Campus Assistant</h1>
        <p>Sri Siddhartha Institute of Technology</p>
        <p>
        Your AI assistant for campus regulations,
        attendance, examinations and hostel rules.
        </p>
    </div>
    """)

    gr.Markdown("### 💬 Ask your campus question")

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(
            height=430,
            show_copy_button=True,
            elem_id="chat-area"
        ),
        textbox=gr.Textbox(
            placeholder="Type your question here...",
            lines=2,
            max_lines=4
        ),
        examples=[
            "What are the attendance rules?",
            "What are the examination regulations?",
            "What are the hostel visiting hours?"
        ]
    )

    gr.HTML("""
    <div id="footer">
        SSIT Campus Assistant · AI-powered campus information
        <br>
        Always verify important regulations against official documents.
    </div>
    """)

demo.launch(share=True)


/tmp/ipykernel_2295/3350954655.py:43: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


TypeError: Chatbot.__init__() got an unexpected keyword argument 'show_copy_button'

In [16]:



demo.launch(import gradio as gr

custom_css = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
    font-family: Arial, sans-serif !important;
}

#hero {
    background: linear-gradient(135deg, #102A43, #176B87);
    color: white;
    padding: 28px;
    border-radius: 18px;
    margin-bottom: 18px;
}

#hero h1 {
    color: white;
    font-size: 30px;
}

#hero p {
    color: #E6F4F7;
    font-size: 15px;
}

#footer {
    text-align: center;
    color: #64748B;
    font-size: 12px;
    padding: 15px;
}
"""

def chatbot(message, history):
    return answer_question(message)

with gr.Blocks() as demo:

    gr.HTML("""
    <div id="hero">
        <h1>🎓 SSIT Campus Assistant</h1>
        <p>Sri Siddhartha Institute of Technology</p>
        <p>
        Your AI assistant for attendance, examinations,
        and hostel regulations.
        </p>
    </div>
    """)

    gr.Markdown("### 💬 Ask your campus question")

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(height=430),
        textbox=gr.Textbox(
            placeholder="Type your question here...",
            lines=2
        ),
        examples=[
            "What are the attendance rules?",
            "What are the examination regulations?",
            "What are the hostel visiting hours?"
        ]
    )

    gr.HTML("""
    <div id="footer">
        SSIT Campus Assistant · AI-powered campus information
        <br>
        Verify important regulations against official documents.
    </div>
    """)
    share=True,
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0e575cd501ea6f55e2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
import gradio as gr

custom_css = """
.gradio-container {
    max-width: 100% !important;
    width: 100% !important;
    min-height: 100vh !important;
    margin: 0 !important;
    padding: 24px !important;
    font-family: Arial, sans-serif !important;
    /* Dark olive transparent overlay combined with your campus image */
    background-image: linear-gradient(rgba(30, 38, 24, 0.78), rgba(30, 38, 24, 0.78)), url('campus.jpg');
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
}

#hero {
    background: linear-gradient(135deg, #2D3823, #3A4D23);
    color: #D4AF37;
    padding: 20px 28px;
    border-radius: 14px;
    border: 1px solid #D4AF37;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    text-align: center;
    margin-bottom: 16px;
}

#hero h1 {
    color: #D4AF37;
    font-size: 28px;
    margin-bottom: 6px;
}

#hero p {
    color: #E6C687;
    font-size: 14px;
    margin: 3px 0;
}

/* Full length bottom rectangular container for chat interface */
.chat-box-wrapper {
    background: rgba(35, 45, 28, 0.92) !important;
    border: 1px solid #D4AF37 !important;
    border-radius: 14px !important;
    padding: 16px !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.6) !important;
    width: 100% !important;
}

.chat-box-wrapper h3 {
    color: #D4AF37 !important;
    margin-bottom: 10px;
}

/* Customizing Submit/Send button in Gold */
button.primary {
    background: #D4AF37 !important;
    color: #1E2618 !important;
    font-weight: bold !important;
    border: none !important;
}

button.primary:hover {
    background: #C5A059 !important;
}

#footer {
    text-align: center;
    color: #E6C687;
    font-size: 12px;
    padding: 10px;
    margin-top: 10px;
}
"""

def answer_question(message):
    # TODO: Replace with your actual RAG / LLM document search logic
    return f"This is a placeholder response for: '{message}'. Connect your official SSIT documentation here!"

def chatbot(message, history):
    return answer_question(message)

with gr.Blocks(css=custom_css) as demo:

    # Top Hero Section
    gr.HTML("""
    <div id="hero">
        <h1>🎓 SSIT Campus Assistant</h1>
        <p>Sri Siddhartha Institute of Technology</p>
        <p>Your AI assistant for attendance, examinations, and hostel regulations.</p>
    </div>
    """)

    # Bottom Full-Length Rectangular Chat Interface
    with gr.Column(elem_classes=["chat-box-wrapper"]):
        gr.Markdown("### 💬 Ask your campus question")
        gr.ChatInterface(
            fn=chatbot,
            chatbot=gr.Chatbot(height=280),
            textbox=gr.Textbox(
                placeholder="Type your question here...",
                lines=2
            ),
            examples=[
                "What are the attendance rules?",
                "What are the examination regulations?",
                "What are the hostel visiting hours?"
            ]
        )

    # Footer Section
    gr.HTML("""
    <div id="footer">
        SSIT Campus Assistant · AI-powered campus information
        <br>
        Verify important regulations against official documents.
    </div>
    """)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

/tmp/ipykernel_2295/599615008.py:87: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f37870919264ab17f5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c575302939df19c998.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://4c70150f808cc5791e.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://0fa225f360525fcd9e.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://e2f09d5dde25c8b547.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://0e575cd501ea6f55e2.gradio.live
Killing tunnel 127.0.0.1:7864 <> https://f37870919264ab17f5.gradio.live


In [18]:
import base64
import gradio as gr

# Automatically load and encode your uploaded campus globe photo
image_path = "image_522376.jpg"
try:
    with open(image_path, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode()
    bg_css = f"background-image: linear-gradient(rgba(30, 38, 24, 0.65), rgba(30, 38, 24, 0.65)), url('data:image/jpeg;base64,{encoded_image}');"
except Exception as e:
    # Fallback background color if image isn't found yet
    bg_css = "background-color: #2D3823;"

custom_css = f"""
.gradio-container {{
    max-width: 100% !important;
    width: 100% !important;
    min-height: 100vh !important;
    margin: 0 !important;
    padding: 24px !important;
    font-family: Arial, sans-serif !important;
    {bg_css}
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
}}

#hero {{
    background: linear-gradient(135deg, #2D3823, #3A4D23);
    color: #D4AF37;
    padding: 20px 28px;
    border-radius: 14px;
    border: 1px solid #D4AF37;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    text-align: center;
    margin-bottom: 16px;
}}

#hero h1 {{
    color: #D4AF37;
    font-size: 28px;
    margin-bottom: 6px;
}}

#hero p {{
    color: #E6C687;
    font-size: 14px;
    margin: 3px 0;
}}

/* Full-length bottom rectangular container for chat interface */
.chat-box-wrapper {{
    background: rgba(30, 38, 24, 0.90) !important;
    border: 1px solid #D4AF37 !important;
    border-radius: 14px !important;
    padding: 16px !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.6) !important;
    width: 100% !important;
}}

.chat-box-wrapper h3 {{
    color: #D4AF37 !important;
    margin-bottom: 10px;
}}

/* Customizing Submit button in Gold */
button.primary {{
    background: #D4AF37 !important;
    color: #1E2618 !important;
    font-weight: bold !important;
    border: none !important;
}}

button.primary:hover {{
    background: #C5A059 !important;
}}

#footer {{
    text-align: center;
    color: #E6C687;
    font-size: 12px;
    padding: 10px;
    margin-top: 10px;
}}
"""

def answer_question(message):
    # TODO: Replace with your actual RAG / LLM document search logic
    return f"This is a placeholder response for: '{message}'. Connect your official SSIT documentation here!"

def chatbot(message, history):
    return answer_question(message)

with gr.Blocks(css=custom_css) as demo:

    # Top Hero Section
    gr.HTML("""
    <div id="hero">
        <h1>🎓 SSIT Campus Assistant</h1>
        <p>Sri Siddhartha Institute of Technology</p>
        <p>Your AI assistant for attendance, examinations, and hostel regulations.</p>
    </div>
    """)

    # Bottom Full-Length Rectangular Chat Interface
    with gr.Column(elem_classes=["chat-box-wrapper"]):
        gr.Markdown("### 💬 Ask your campus question")
        gr.ChatInterface(
            fn=chatbot,
            chatbot=gr.Chatbot(height=260),
            textbox=gr.Textbox(
                placeholder="Type your question here...",
                lines=2
            ),
            examples=[
                "What are the attendance rules?",
                "What are the examination regulations?",
                "What are the hostel visiting hours?"
            ]
        )

    # Footer Section
    gr.HTML("""
    <div id="footer">
        SSIT Campus Assistant · AI-powered campus information
        <br>
        Verify important regulations against official documents.
    </div>
    """)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

/tmp/ipykernel_2295/2764902353.py:97: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e85fbfbd2d4bd5b44c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://e85fbfbd2d4bd5b44c.gradio.live


In [19]:
import base64
import os
import gradio as gr

# Automatically find the uploaded campus/globe image in the Colab directory
image_path = "image_522376.jpg"
if not os.path.exists(image_path):
    # Look for any uploaded image file in the current directory as a fallback
    image_candidates = [f for f in os.listdir('.') if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if image_candidates:
        image_path = image_candidates[0]
        print(f"⚠️ Exact file name not found. Automatically using: '{image_path}'")

# Encode the image into base64 for seamless background loading
try:
    with open(image_path, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode()
    bg_css = f"background-image: linear-gradient(rgba(30, 38, 24, 0.65), rgba(30, 38, 24, 0.65)), url('data:image/jpeg;base64,{encoded_image}');"
    print(f"✅ Background globe image loaded successfully!")
except Exception as e:
    print(f"❌ Error loading background image: {e}")
    bg_css = "background-color: #2D3823;"

custom_css = f"""
.gradio-container {{
    max-width: 100% !important;
    width: 100% !important;
    min-height: 100vh !important;
    margin: 0 !important;
    padding: 24px !important;
    font-family: Arial, sans-serif !important;
    {bg_css}
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
}}

#hero {{
    background: linear-gradient(135deg, #2D3823, #3A4D23);
    color: #D4AF37;
    padding: 20px 28px;
    border-radius: 14px;
    border: 1px solid #D4AF37;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    text-align: center;
    margin-bottom: 16px;
}}

#hero h1 {{
    color: #D4AF37;
    font-size: 28px;
    margin-bottom: 6px;
}}

#hero p {{
    color: #E6C687;
    font-size: 14px;
    margin: 3px 0;
}}

/* Full-length bottom rectangular container for chat interface */
.chat-box-wrapper {{
    background: rgba(30, 38, 24, 0.90) !important;
    border: 1px solid #D4AF37 !important;
    border-radius: 14px !important;
    padding: 16px !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.6) !important;
    width: 100% !important;
}}

.chat-box-wrapper h3 {{
    color: #D4AF37 !important;
    margin-bottom: 10px;
}}

/* Customizing Submit button in Gold */
button.primary {{
    background: #D4AF37 !important;
    color: #1E2618 !important;
    font-weight: bold !important;
    border: none !important;
}}

button.primary:hover {{
    background: #C5A059 !important;
}}

#footer {{
    text-align: center;
    color: #E6C687;
    font-size: 12px;
    padding: 10px;
    margin-top: 10px;
}}
"""

def answer_question(message):
    # TODO: Replace with your actual RAG / LLM document search logic
    return f"This is a placeholder response for: '{message}'. Connect your official SSIT documentation here!"

def chatbot(message, history):
    return answer_question(message)

with gr.Blocks(css=custom_css) as demo:

    # Top Hero Section (SSI-D Campus Assistant)
    gr.HTML("""
    <div id="hero">
        <h1>🎓 SSIT Campus Assistant</h1>
        <p>Sri Siddhartha Institute of Technology</p>
        <p>Your AI assistant for attendance, examinations, and hostel regulations.</p>
    </div>
    """)

    # Bottom Full-Length Rectangular Chat Interface
    with gr.Column(elem_classes=["chat-box-wrapper"]):
        gr.Markdown("### 💬 Ask your campus question")
        gr.ChatInterface(
            fn=chatbot,
            chatbot=gr.Chatbot(height=260),
            textbox=gr.Textbox(
                placeholder="Type your question here...",
                lines=2
            ),
            examples=[
                "What are the attendance rules?",
                "What are the examination regulations?",
                "What are the hostel visiting hours?"
            ]
        )

    # Footer Section
    gr.HTML("""
    <div id="footer">
        SSIT Campus Assistant · AI-powered campus information
        <br>
        Verify important regulations against official documents.
    </div>
    """)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

❌ Error loading background image: [Errno 2] No such file or directory: 'image_522376.jpg'


/tmp/ipykernel_2295/117423354.py:107: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://67ba18704914ea39fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://67ba18704914ea39fa.gradio.live


In [20]:
import base64
import os
import gradio as gr

# Automatically find or check for your uploaded campus/globe image in Colab
image_path = "image_522376.jpg"
bg_css = ""

if os.path.exists(image_path):
    try:
        with open(image_path, "rb") as image_file:
            encoded_image = base64.b64encode(image_file.read()).decode()
        bg_css = f"background-image: linear-gradient(rgba(30, 38, 24, 0.65), rgba(30, 38, 24, 0.65)), url('data:image/jpeg;base64,{encoded_image}');"
        print("✅ Background globe image loaded successfully!")
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        bg_css = "background-color: #2D3823;"
else:
    # Look for any other image uploaded in Colab
    image_candidates = [f for f in os.listdir('.') if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if image_candidates:
        image_path = image_candidates[0]
        with open(image_path, "rb") as image_file:
            encoded_image = base64.b64encode(image_file.read()).decode()
        bg_css = f"background-image: linear-gradient(rgba(30, 38, 24, 0.65), rgba(30, 38, 24, 0.65)), url('data:image/jpeg;base64,{encoded_image}');"
        print(f"⚠️ 'image_522376.jpg' not found. Automatically using found image: '{image_path}'")
    else:
        print("❌ WARNING: No image file found in Colab files! Please upload your globe photo to the left sidebar.")
        bg_css = "background-color: #2D3823;"

custom_css = f"""
.gradio-container {{
    max-width: 100% !important;
    width: 100% !important;
    min-height: 100vh !important;
    margin: 0 !important;
    padding: 24px !important;
    font-family: Arial, sans-serif !important;
    {bg_css}
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
}}

#hero {{
    background: linear-gradient(135deg, #2D3823, #3A4D23);
    color: #D4AF37;
    padding: 20px 28px;
    border-radius: 14px;
    border: 1px solid #D4AF37;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    text-align: center;
    margin-bottom: 16px;
}}

#hero h1 {{
    color: #D4AF37;
    font-size: 28px;
    margin-bottom: 6px;
}}

#hero p {{
    color: #E6C687;
    font-size: 14px;
    margin: 3px 0;
}}

/* Full-length bottom rectangular container for chat interface */
.chat-box-wrapper {{
    background: rgba(30, 38, 24, 0.90) !important;
    border: 1px solid #D4AF37 !important;
    border-radius: 14px !important;
    padding: 16px !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.6) !important;
    width: 100% !important;
}}

.chat-box-wrapper h3 {{
    color: #D4AF37 !important;
    margin-bottom: 10px;
}}

/* Customizing Submit button in Gold */
button.primary {{
    background: #D4AF37 !important;
    color: #1E2618 !important;
    font-weight: bold !important;
    border: none !important;
}}

button.primary:hover {{
    background: #C5A059 !important;
}}

#footer {{
    text-align: center;
    color: #E6C687;
    font-size: 12px;
    padding: 10px;
    margin-top: 10px;
}}
"""

def answer_question(message):
    # TODO: Replace with your actual RAG / LLM document search logic
    return f"This is a placeholder response for: '{message}'. Connect your official SSIT documentation here!"

def chatbot(message, history):
    return answer_question(message)

# Clean Blocks definition (css removed from here for Gradio 6.0 compliance)
with gr.Blocks() as demo:

    # Top Hero Section
    gr.HTML("""
    <div id="hero">
        <h1>🎓 SSIT Campus Assistant</h1>
        <p>Sri Siddhartha Institute of Technology</p>
        <p>Your AI assistant for attendance, examinations, and hostel regulations.</p>
    </div>
    """)

    # Bottom Full-Length Rectangular Chat Interface
    with gr.Column(elem_classes=["chat-box-wrapper"]):
        gr.Markdown("### 💬 Ask your campus question")
        gr.ChatInterface(
            fn=chatbot,
            chatbot=gr.Chatbot(height=260),
            textbox=gr.Textbox(
                placeholder="Type your question here...",
                lines=2
            ),
            examples=[
                "What are the attendance rules?",
                "What are the examination regulations?",
                "What are the hostel visiting hours?"
            ]
        )

    # Footer Section
    gr.HTML("""
    <div id="footer">
        SSIT Campus Assistant · AI-powered campus information
        <br>
        Verify important regulations against official documents.
    </div>
    """)

if __name__ == "__main__":
    # css passed correctly into launch() for Gradio 6.0
    demo.launch(share=True, debug=True, css=custom_css)

❌ WARNING: No image file found in Colab files! Please upload your globe photo to the left sidebar.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://58334a4a57c52dcefe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://58334a4a57c52dcefe.gradio.live


In [21]:
import base64
import os
import gradio as gr

# Check both local and absolute Colab paths for your globe image
image_paths = ["image_522376.jpg", "/content/image_522376.jpg"]
image_path = None

for path in image_paths:
    if os.path.exists(path):
        image_path = path
        break

bg_css = ""
if image_path:
    try:
        with open(image_path, "rb") as image_file:
            encoded_image = base64.b64encode(image_file.read()).decode()
        bg_css = f"background-image: linear-gradient(rgba(30, 38, 24, 0.65), rgba(30, 38, 24, 0.65)), url('data:image/jpeg;base64,{encoded_image}');"
        print(f"✅ Background globe image loaded successfully from '{image_path}'!")
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        bg_css = "background-color: #2D3823;"
else:
    print("❌ WARNING: Globe image file not found! Please make sure it's uploaded.")
    bg_css = "background-color: #2D3823;"

custom_css = f"""
.gradio-container {{
    max-width: 100% !important;
    width: 100% !important;
    min-height: 100vh !important;
    margin: 0 !important;
    padding: 24px !important;
    font-family: Arial, sans-serif !important;
    {bg_css}
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
}}

#hero {{
    background: linear-gradient(135deg, #2D3823, #3A4D23);
    color: #D4AF37;
    padding: 20px 28px;
    border-radius: 14px;
    border: 1px solid #D4AF37;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    text-align: center;
    margin-bottom: 16px;
}}

#hero h1 {{
    color: #D4AF37;
    font-size: 28px;
    margin-bottom: 6px;
}}

#hero p {{
    color: #E6C687;
    font-size: 14px;
    margin: 3px 0;
}}

/* Full-length bottom rectangular container for chat interface */
.chat-box-wrapper {{
    background: rgba(30, 38, 24, 0.90) !important;
    border: 1px solid #D4AF37 !important;
    border-radius: 14px !important;
    padding: 16px !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.6) !important;
    width: 100% !important;
}}

.chat-box-wrapper h3 {{
    color: #D4AF37 !important;
    margin-bottom: 10px;
}}

/* Customizing Submit button in Gold */
button.primary {{
    background: #D4AF37 !important;
    color: #1E2618 !important;
    font-weight: bold !important;
    border: none !important;
}}

button.primary:hover {{
    background: #C5A059 !important;
}}

#footer {{
    text-align: center;
    color: #E6C687;
    font-size: 12px;
    padding: 10px;
    margin-top: 10px;
}}
"""

def answer_question(message):
    # TODO: Replace with your actual RAG / LLM document search logic
    return f"This is a placeholder response for: '{message}'. Connect your official SSIT documentation here!"

def chatbot(message, history):
    return answer_question(message)

with gr.Blocks() as demo:

    # Top Hero Section
    gr.HTML("""
    <div id="hero">
        <h1>🎓 SSIT Campus Assistant</h1>
        <p>Sri Siddhartha Institute of Technology</p>
        <p>Your AI assistant for attendance, examinations, and hostel regulations.</p>
    </div>
    """)

    # Bottom Full-Length Rectangular Chat Interface
    with gr.Column(elem_classes=["chat-box-wrapper"]):
        gr.Markdown("### 💬 Ask your campus question")
        gr.ChatInterface(
            fn=chatbot,
            chatbot=gr.Chatbot(height=260),
            textbox=gr.Textbox(
                placeholder="Type your question here...",
                lines=2
            ),
            examples=[
                "What are the attendance rules?",
                "What are the examination regulations?",
                "What are the hostel visiting hours?"
            ]
        )

    # Footer Section
    gr.HTML("""
    <div id="footer">
        SSIT Campus Assistant · AI-powered campus information
        <br>
        Verify important regulations against official documents.
    </div>
    """)

if __name__ == "__main__":
    demo.launch(share=True, debug=True, css=custom_css)

❌ WARNING: Globe image file not found! Please make sure it's uploaded.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3291d12b7521b67543.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://3291d12b7521b67543.gradio.live


In [23]:
import base64
import os
import glob
import gradio as gr

# Automatically search for any image uploaded in Colab (/content/ or current directory)
print("--- Scanning Colab files for your globe photo ---")
image_path = None
search_patterns = ["/content/*.jpg", "/content/*.jpeg", "/content/*.png", "./*.jpg", "./*.jpeg", "./*.png"]

for pattern in search_patterns:
    matches = glob.glob(pattern)
    # Filter out sample files if any exist
    matches = [m for m in matches if "sample_data" not in m]
    if matches:
        image_path = matches[0]
        break

bg_css = ""
if image_path and os.path.exists(image_path):
    try:
        with open(image_path, "rb") as image_file:
            encoded_image = base64.b64encode(image_file.read()).decode()
        bg_css = f"background-image: linear-gradient(rgba(30, 38, 24, 0.65), rgba(30, 38, 24, 0.65)), url('data:image/jpeg;base64,{encoded_image}');"
        print(f"✅ SUCCESS: Background globe image loaded from: {image_path}")
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        bg_css = "background-color: #2D3823;"
else:
    print("❌ WARNING: No image file detected in Colab files! Please check the folder icon on the left sidebar and upload your globe photo.")
    bg_css = "background-color: #2D3823;"

custom_css = f"""
.gradio-container {{
    max-width: 100% !important;
    width: 100% !important;
    min-height: 100vh !important;
    margin: 0 !important;
    padding: 24px !important;
    font-family: Arial, sans-serif !important;
    {bg_css}
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
}}

#hero {{
    background: linear-gradient(135deg, #2D3823, #3A4D23);
    color: #D4AF37;
    padding: 20px 28px;
    border-radius: 14px;
    border: 1px solid #D4AF37;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    text-align: center;
    margin-bottom: 16px;
}}

#hero h1 {{
    color: #D4AF37;
    font-size: 28px;
    margin-bottom: 6px;
}}

#hero p {{
    color: #E6C687;
    font-size: 14px;
    margin: 3px 0;
}}

/* Full-length bottom rectangular container for chat interface */
.chat-box-wrapper {{
    background: rgba(30, 38, 24, 0.90) !important;
    border: 1px solid #D4AF37 !important;
    border-radius: 14px !important;
    padding: 16px !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.6) !important;
    width: 100% !important;
}}

.chat-box-wrapper h3 {{
    color: #D4AF37 !important;
    margin-bottom: 10px;
}}

/* Customizing Submit button in Gold */
button.primary {{
    background: #D4AF37 !important;
    color: #1E2618 !important;
    font-weight: bold !important;
    border: none !important;
}}

button.primary:hover {{
    background: #C5A059 !important;
}}

#footer {{
    text-align: center;
    color: #E6C687;
    font-size: 12px;
    padding: 10px;
    margin-top: 10px;
}}
"""

def answer_question(message):
    # TODO: Replace with your actual RAG / LLM document search logic
    return f"This is a placeholder response for: '{message}'. Connect your official SSIT documentation here!"

def chatbot(message, history):
    return answer_question(message)

with gr.Blocks() as demo:

    # Top Hero Section
    gr.HTML("""
    <div id="hero">
        <h1>🎓 SSIT Campus Assistant</h1>
        <p>Sri Siddhartha Institute of Technology</p>
        <p>Your AI assistant for attendance, examinations, and hostel regulations.</p>
    </div>
    """)

    # Bottom Full-Length Rectangular Chat Interface
    with gr.Column(elem_classes=["chat-box-wrapper"]):
        gr.Markdown("### 💬 Ask your campus question")
        gr.ChatInterface(
            fn=chatbot,
            chatbot=gr.Chatbot(height=260),
            textbox=gr.Textbox(
                placeholder="Type your question here...",
                lines=2
            ),
            examples=[
                "What are the attendance rules?",
                "What are the examination regulations?",
                "What are the hostel visiting hours?"
            ]
        )

    # Footer Section
    gr.HTML("""
    <div id="footer">
        SSIT Campus Assistant · AI-powered campus information
        <br>
        Verify important regulations against official documents.
    </div>
    """)

if __name__ == "__main__":
    demo.launch(share=True, debug=True, css=custom_css)

--- Scanning Colab files for your globe photo ---
❌ WARNING: No image file detected in Colab files! Please check the folder icon on the left sidebar and upload your globe photo.


KeyboardInterrupt: 

In [24]:

import gradio as gr

# Replace this URL with your college globe photo URL
BG_IMAGE = "https://images.unsplash.com/photo-1562774053-701939374585?auto=format&fit=crop&w=1920&q=85"

custom_css = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    font-family: Arial, sans-serif !important;
}

#main {
    background: linear-gradient(
        rgba(8, 25, 42, 0.77),
        rgba(8, 25, 42, 0.86)
    ), url("BG_IMAGE_PLACEHOLDER");

    background-size: cover;
    background-position: center;
    background-attachment: fixed;
    border-radius: 18px;
    padding: 24px;
    min-height: 90vh;
}

#header {
    text-align: center;
    color: white;
    padding: 20px 10px;
    border-bottom: 1px solid rgba(255,255,255,0.3);
    margin-bottom: 20px;
}

#header h1 {
    font-size: 32px;
    margin-bottom: 8px;
}

#header p {
    font-size: 15px;
    color: #e4edf5;
}

#chat {
    background: rgba(255,255,255,0.96);
    border-radius: 14px;
    padding: 10px;
}

#question button {
    border-radius: 8px !important;
    font-size: 13px !important;
}

#footer {
    color: white;
    text-align: center;
    font-size: 12px;
    padding-top: 16px;
}

footer {
    display: none !important;
}
"""

custom_css = custom_css.replace(
    "BG_IMAGE_PLACEHOLDER", BG_IMAGE
)

def reply(message, history):
    if not message.strip():
        return "Please enter your question."

    # UI demo only: connect your RAG model here
    return (
        "Welcome to the SSIT Campus Assistant.\n\n"
        "Your question has been received: "
        + message
        + "\n\nConnect your college knowledge base "
          "to provide accurate campus information."
    )

with gr.Blocks(
    css=custom_css,
    title="SSIT Campus Assistant",
    theme=gr.themes.Soft(
        primary_hue="blue",
        neutral_hue="slate"
    )
) as demo:

    with gr.Column(elem_id="main"):

        gr.HTML("""
        <div id="header">
            <h1>SSIT Campus Assistant</h1>
            <p>Sri Siddhartha Institute of Technology</p>
            <p>Your official campus information assistant</p>
        </div>
        """)

        gr.ChatInterface(
            fn=reply,
            chatbot=gr.Chatbot(
                type="messages",
                height=390,
                show_copy_button=True,
                elem_id="chat"
            ),
            textbox=gr.Textbox(
                placeholder="Ask about admissions, departments, fees...",
                lines=1,
                max_lines=3,
                container=False
            ),
            examples=[
                "What courses are offered?",
                "How can I apply for admission?",
                "Where is the library?",
                "What are the college office hours?"
            ],
            submit_btn="Send",
            clear_btn="Clear chat"
        )

        gr.HTML("""
        <div id="footer">
            SSIT Campus Assistant | Student Support & Campus Information
        </div>
        """)

demo.launch(share=True, debug=True)


/tmp/ipykernel_2295/2322666508.py:85: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


TypeError: Chatbot.__init__() got an unexpected keyword argument 'type'

In [25]:
!pip install -q -U gradio

In [28]:

import gradio as gr

# ---------- DARK OLIVE & GOLD COLLEGE THEME ----------

custom_css = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
    font-family: Georgia, 'Times New Roman', serif !important;
    background: #F5F2E8 !important;
}

#hero {
    background: linear-gradient(135deg, #252D18, #414A29);
    color: #F7F0DC;
    padding: 32px 24px;
    border: 2px solid #B89A4A;
    border-radius: 12px;
    margin-bottom: 22px;
    text-align: center;
}

#hero .emblem {
    color: #D6B65B;
    font-size: 36px;
}

#hero h1 {
    color: #E2C36B !important;
    font-size: 30px;
    letter-spacing: 1px;
}

#hero p {
    color: #F5F0E3 !important;
    font-size: 15px;
}

.gold-line {
    height: 2px;
    max-width: 220px;
    background: #C5A44D;
    margin: 18px auto;
}

#chatbot {
    border: 1px solid #B89A4A !important;
    border-radius: 12px !important;
    background: #FFFDF7 !important;
}

#footer {
    text-align: center;
    color: #62664D;
    font-family: Arial, sans-serif;
    font-size: 12px;
    padding: 20px 10px;
    border-top: 1px solid #C8B77D;
    margin-top: 20px;
}

footer {
    display: none !important;
}
"""

# ---------- CHATBOT FUNCTION ----------

def chatbot(message, history):
    try:
        return answer_question(message)
    except NameError:
        return (
            "Welcome to the SSIT Campus Assistant. "
            "Your question is: " + message +
            "\n\nThis is a demo response. Connect your "
            "college RAG pipeline for accurate campus information."
        )
    except Exception as e:
        return f"Sorry, an error occurred: {str(e)}"

# ---------- COLLEGE CHATBOT INTERFACE ----------

with gr.Blocks(title="SSIT | Campus Assistant") as demo:

    gr.HTML("""
    <div id="hero">
        <div class="emblem">✦</div>
        <p>SRI SIDDHARTHA INSTITUTE OF TECHNOLOGY</p>
        <div class="gold-line"></div>
        <h1>SSIT CAMPUS ASSISTANT</h1>
        <p>OFFICIAL STUDENT INFORMATION PORTAL</p>
        <p>Academic Guidance · Campus Services · Student Support</p>
    </div>
    """)

    gr.Markdown("""
    ## Welcome to the Campus Help Desk

    Get assistance with college information, attendance,
    examinations, departments and hostel facilities.
    """)

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(
            height=420,
            elem_id="chatbot"
        ),
        textbox=gr.Textbox(
            placeholder="Type your campus question here...",
            lines=2
        ),
        examples=[
            "What are the attendance requirements?",
            "What are the examination regulations?",
            "What are the hostel visiting hours?",
            "Where can I find department information?"
        ],
        submit_btn="Ask Assistant"
    )

    gr.HTML("""
    <div id="footer">
        SSIT CAMPUS ASSISTANT<br>
        Academic Information and Student Support<br>
        Verify important regulations with official college sources.
    </div>
    """)

# ---------- LAUNCH APP & GENERATE URL ----------

demo.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="lime",
        secondary_hue="amber",
        neutral_hue="stone"
    ),
    css=custom_css
)


/tmp/ipykernel_2295/2268160188.py:104: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4185b950eaddf204ef.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [30]:

!pip install -q -U gradio

import gradio as gr

# ==========================================
# SSIT CAMPUS ASSISTANT
# Theme: Dark Olive + Gold
# ==========================================

LOGO_URL = (
    "https://ssit.edu.in/img/"
    "sri-siddhartha-institute-of-technology-ssit-logo.png"
)

custom_css = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
    font-family: Georgia, 'Times New Roman', serif !important;
    background: #F5F2E8 !important;
}

/* College header */
#hero {
    background: linear-gradient(135deg, #252D18, #414A29);
    color: #F7F0DC;
    padding: 32px 24px;
    border: 2px solid #B89A4A;
    border-radius: 12px;
    margin-bottom: 22px;
    text-align: center;
}

#college-logo {
    width: 115px;
    height: 115px;
    object-fit: contain;
    background: white;
    border: 2px solid #D6B65B;
    border-radius: 10px;
    padding: 5px;
    margin-bottom: 12px;
}

#hero h1 {
    color: #E2C36B !important;
    font-size: 30px;
    letter-spacing: 1px;
    margin: 12px 0;
}

#hero p {
    color: #F5F0E3 !important;
    font-size: 15px;
    margin: 8px 0;
}

.gold-line {
    height: 2px;
    max-width: 220px;
    background: #C5A44D;
    margin: 18px auto;
}

/* Chat panel */
#chatbot {
    border: 1px solid #B89A4A !important;
    border-radius: 12px !important;
    background: #FFFDF7 !important;
}

/* Footer */
#footer {
    text-align: center;
    color: #62664D;
    font-family: Arial, sans-serif;
    font-size: 12px;
    padding: 20px 10px;
    border-top: 1px solid #C8B77D;
    margin-top: 20px;
}

footer {
    display: none !important;
}
"""

# ==========================================
# CHATBOT FUNCTION
# ==========================================

def chatbot(message, history):
    if not message.strip():
        return "Please enter your campus-related question."

    # Use your existing RAG pipeline if available
    try:
        return answer_question(message)
    except NameError:
        return (
            "Welcome to the SSIT Campus Assistant!\\n\\n"
            "Your question: " + message + "\\n\\n"
            "This is a demonstration response. "
            "Connect your campus knowledge base to provide "
            "accurate answers from official college documents."
        )
    except Exception as error:
        print("Chatbot error:", error)
        return (
            "Sorry, I couldn't process that question. "
            "Please try again."
        )

# ==========================================
# BUILD INTERFACE
# ==========================================

with gr.Blocks(title="SSIT | Campus Assistant") as demo:

    gr.HTML(f"""
    <div id="hero">
        <img
            id="college-logo"
            src="{LOGO_URL}"
            alt="Sri Siddhartha Institute of Technology Logo"
        >

        <p>SRI SIDDHARTHA INSTITUTE OF TECHNOLOGY</p>

        <div class="gold-line"></div>

        <h1>SSIT CAMPUS ASSISTANT</h1>

        <p>OFFICIAL STUDENT INFORMATION PORTAL</p>

        <p>
            Academic Guidance · Campus Services · Student Support
        </p>
    </div>
    """)

    gr.Markdown("""
    ## Welcome to the Campus Help Desk

    Ask questions about college academics, attendance,
    examinations, departments and hostel facilities.
    """)

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(
            height=420,
            elem_id="chatbot"
        ),
        textbox=gr.Textbox(
            placeholder="Type your campus question here...",
            lines=2
        ),
        examples=[
            "What are the attendance requirements?",
            "What are the examination regulations?",
            "What are the hostel visiting hours?",
            "Where can I find department information?"
        ],
        submit_btn="Ask Assistant"
    )

    gr.HTML("""
    <div id="footer">
        SSIT CAMPUS ASSISTANT<br>
        Academic Information and Student Support<br>
        Please verify important regulations with official college sources.
    </div>
    """)

# ==========================================
# LAUNCH AND GENERATE PUBLIC URL
# ==========================================

demo.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="lime",
        secondary_hue="amber",
        neutral_hue="stone"
    ),
    css=custom_css
)




/tmp/ipykernel_2295/3771573198.py:149: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2af8d2ec4e6a84ef63.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [31]:

!pip install -q -U gradio

import os
import gradio as gr

# ==========================================
# 1. SSIT LOGO — YOUR EXACT FILE NAME
# ==========================================
LOGO_PATH = "/content/ssit_logo.png.jpeg"

if not os.path.exists(LOGO_PATH):
    raise FileNotFoundError(
        f"Logo not found at {LOGO_PATH}. "
        "Check the filename in the Colab Files panel."
    )

# ==========================================
# 2. PROFESSIONAL OLIVE & GOLD DESIGN
# ==========================================
custom_css = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
    background: #F5F2E8 !important;
    font-family: Arial, sans-serif !important;
}

body {
    background: #F5F2E8 !important;
}

#hero {
    background: linear-gradient(135deg, #252D18, #414A29);
    color: #F5F2E8;
    padding: 32px 24px;
    border: 2px solid #B89A4A;
    border-radius: 18px;
    text-align: center;
    margin: 12px 0 28px 0;
    box-shadow: 0 5px 16px rgba(0,0,0,0.12);
}

#college-logo {
    width: 115px !important;
    height: 115px !important;
    margin: 0 auto 18px auto !important;
    background: white !important;
    border: 2px solid #D6B65B !important;
    border-radius: 12px !important;
    overflow: hidden;
}

#college-logo img {
    width: 100% !important;
    height: 100% !important;
    object-fit: contain !important;
}

.college-name {
    color: #F5F2E8 !important;
    font-size: 19px !important;
    font-weight: 600 !important;
    text-align: center !important;
    letter-spacing: 0.5px;
}

.gold-line {
    width: 220px;
    height: 3px;
    background: #D6B65B;
    margin: 18px auto;
    border-radius: 5px;
}

.main-title {
    color: #D6B65B !important;
    font-size: 36px !important;
    font-weight: 800 !important;
    text-align: center !important;
    margin: 12px 0 !important;
}

.subtitle {
    color: white !important;
    font-size: 18px !important;
    text-align: center !important;
    margin: 12px 0 !important;
}

.tagline {
    color: #F5F2E8 !important;
    font-size: 16px !important;
    text-align: center !important;
}

#welcome {
    color: #252D18 !important;
}

#chatbot {
    border: 1px solid #B89A4A !important;
    border-radius: 16px !important;
    background: #FFFCF5 !important;
}

button.primary {
    background: #414A29 !important;
    color: white !important;
    border: 1px solid #B89A4A !important;
}

button.primary:hover {
    background: #252D18 !important;
}

footer {
    display: none !important;
}

@media (max-width: 600px) {
    #hero {
        padding: 24px 14px;
    }

    .main-title {
        font-size: 27px !important;
    }

    .college-name {
        font-size: 15px !important;
    }
}
"""

# ==========================================
# 3. CHATBOT FUNCTION
# ==========================================
def chatbot(message, history):
    question = message.strip()

    if not question:
        return "Please enter your campus-related question."

    q = question.lower()

    if any(word in q for word in ["hello", "hi", "hey"]):
        return (
            "Hello! Welcome to the SSIT Campus Assistant. "
            "How can I help you today?"
        )

    if "attendance" in q:
        return (
            "For attendance rules and your current attendance, "
            "please check the official college portal or contact "
            "your department office."
        )

    if any(word in q for word in ["exam", "examination", "timetable"]):
        return (
            "Please check official SSIT notices or contact your "
            "department for the latest examination timetable."
        )

    if any(word in q for word in ["hostel", "mess", "accommodation"]):
        return (
            "For hostel accommodation, mess timings and facilities, "
            "please contact the SSIT hostel office."
        )

    if any(word in q for word in ["department", "course", "branch"]):
        return (
            "Please mention the department or course you want "
            "information about. Verified college information can "
            "be connected to provide more detailed answers."
        )

    if any(word in q for word in ["admission", "fee", "fees"]):
        return (
            "For current admission requirements and fee details, "
            "please consult the official SSIT admissions office."
        )

    return (
        "Welcome to the SSIT Campus Assistant! This is the interface "
        "demo. Connect your RAG pipeline and college documents to "
        "answer campus questions using verified information."
    )

# ==========================================
# 4. BUILD THE SSIT CAMPUS ASSISTANT
# ==========================================
with gr.Blocks(title="SSIT | Campus Assistant") as demo:

    with gr.Column(elem_id="hero"):

        gr.Image(
            value=LOGO_PATH,
            show_label=False,
            container=False,
            interactive=False,
            height=115,
            width=115,
            elem_id="college-logo"
        )

        gr.HTML("""
            <div class="college-name">
                SRI SIDDHARTHA INSTITUTE OF TECHNOLOGY
            </div>

            <div class="gold-line"></div>

            <div class="main-title">
                SSIT CAMPUS ASSISTANT
            </div>

            <div class="subtitle">
                OFFICIAL STUDENT INFORMATION PORTAL
            </div>

            <div class="tagline">
                Academic Guidance · Campus Services · Student Support
            </div>
        """)

    gr.Markdown(
        "## Welcome to the Campus Help Desk",
        elem_id="welcome"
    )

    gr.Markdown(
        "Ask questions about college academics, attendance, "
        "examinations, departments and hostel facilities."
    )

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(height=420, elem_id="chatbot"),
        textbox=gr.Textbox(
            placeholder="Type your campus question here...",
            lines=2
        ),
        examples=[
            "How can I get information about attendance?",
            "Where can I find the examination timetable?",
            "Tell me about hostel facilities.",
            "How do I get admission information?"
        ],
        submit_btn="Ask Assistant"
    )

    gr.HTML("""
        <div style="
            text-align:center;
            padding:20px 10px;
            color:#414A29;
            font-size:13px;
        ">
            SSIT Campus Assistant · Student Information Portal
            <br>
            Please verify important information through official
            college sources.
        </div>
    """)

# ==========================================
# 5. LAUNCH THE APPLICATION
# ==========================================
demo.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="lime",
        secondary_hue="amber",
        neutral_hue="stone"
    ),
    css=custom_css
)


FileNotFoundError: Logo not found at /content/ssit_logo.png.jpeg. Check the filename in the Colab Files panel.

In [32]:

# ==============================================
# SSIT CAMPUS ASSISTANT - COMPLETE GOOGLE COLAB CODE
# ==============================================

!pip install -q -U gradio

import os
import gradio as gr
from google.colab import files

# ==============================================
# 1. UPLOAD SSIT LOGO
# ==============================================

print("Upload your SSIT college logo (PNG or JPG).")
uploaded = files.upload()

LOGO_PATH = None

for filename in uploaded.keys():
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        LOGO_PATH = os.path.abspath(filename)
        break

if LOGO_PATH is None:
    raise ValueError(
        "No supported image was uploaded. Please upload a PNG or JPG logo."
    )

print("Logo loaded:", os.path.basename(LOGO_PATH))


# ==============================================
# 2. PROFESSIONAL OLIVE AND GOLD DESIGN
# ==============================================

custom_css = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
    background: #F5F2E8 !important;
    font-family: Arial, sans-serif !important;
}

body {
    background: #F5F2E8 !important;
}

#hero {
    background: linear-gradient(135deg, #252D18, #414A29);
    color: #F5F2E8;
    padding: 32px 24px;
    border: 2px solid #B89A4A;
    border-radius: 18px;
    text-align: center;
    margin: 12px 0 28px 0;
    box-shadow: 0 5px 16px rgba(0,0,0,0.12);
}

#college-logo {
    width: 115px !important;
    height: 115px !important;
    margin: 0 auto 18px auto !important;
    background: white !important;
    border: 2px solid #D6B65B !important;
    border-radius: 12px !important;
    overflow: hidden;
}

#college-logo img {
    width: 100% !important;
    height: 100% !important;
    object-fit: contain !important;
}

.college-name {
    color: #F5F2E8 !important;
    font-size: 19px !important;
    font-weight: 600 !important;
    text-align: center !important;
    letter-spacing: 0.5px;
}

.gold-line {
    width: 220px;
    height: 3px;
    background: #D6B65B;
    margin: 18px auto;
    border-radius: 5px;
}

.main-title {
    color: #D6B65B !important;
    font-size: 36px !important;
    font-weight: 800 !important;
    text-align: center !important;
    margin: 12px 0 !important;
}

.subtitle {
    color: #FFFFFF !important;
    font-size: 18px !important;
    text-align: center !important;
    margin: 12px 0 !important;
}

.tagline {
    color: #F5F2E8 !important;
    font-size: 16px !important;
    text-align: center !important;
}

#welcome {
    color: #252D18 !important;
}

#chatbot {
    border: 1px solid #B89A4A !important;
    border-radius: 16px !important;
    background: #FFFCF5 !important;
}

button.primary {
    background: #414A29 !important;
    color: white !important;
    border: 1px solid #B89A4A !important;
}

button.primary:hover {
    background: #252D18 !important;
}

footer {
    display: none !important;
}

@media (max-width: 600px) {
    #hero {
        padding: 24px 14px;
    }

    .main-title {
        font-size: 27px !important;
    }

    .college-name {
        font-size: 15px !important;
    }
}
"""


# ==============================================
# 3. CHATBOT RESPONSE FUNCTION
# ==============================================

def chatbot(message, history):
    question = message.strip()

    if not question:
        return "Please enter your campus-related question."

    q = question.lower()

    if any(word in q for word in ["hello", "hi", "hey"]):
        return (
            "Hello! Welcome to the SSIT Campus Assistant. "
            "How can I help you today?"
        )

    if "attendance" in q:
        return (
            "For attendance rules and your current attendance, "
            "please check the official college portal or contact "
            "your department office."
        )

    if any(word in q for word in ["exam", "examination", "timetable"]):
        return (
            "For the latest examination timetable, please check "
            "official SSIT notices or contact your department."
        )

    if any(word in q for word in ["hostel", "mess", "accommodation"]):
        return (
            "For hostel accommodation, mess timings and facilities, "
            "please contact the SSIT hostel office."
        )

    if any(word in q for word in ["department", "course", "branch"]):
        return (
            "Please mention the department or course you want "
            "information about. Connect the verified college "
            "knowledge base for detailed answers."
        )

    if any(word in q for word in ["admission", "fee", "fees"]):
        return (
            "For current admission requirements and fee details, "
            "please consult the official SSIT admissions office."
        )

    if any(word in q for word in ["library", "books"]):
        return (
            "For library timings, book availability and library rules, "
            "please check official SSIT library information."
        )

    return (
        "Welcome to the SSIT Campus Assistant! This is a demo "
        "interface. Connect your RAG pipeline and college documents "
        "to answer questions using verified SSIT information."
    )


# ==============================================
# 4. BUILD THE INTERFACE
# ==============================================

with gr.Blocks(title="SSIT | Campus Assistant") as demo:

    with gr.Column(elem_id="hero"):

        gr.Image(
            value=LOGO_PATH,
            show_label=False,
            container=False,
            interactive=False,
            height=115,
            width=115,
            elem_id="college-logo"
        )

        gr.HTML("""
            <div class="college-name">
                SRI SIDDHARTHA INSTITUTE OF TECHNOLOGY
            </div>

            <div class="gold-line"></div>

            <div class="main-title">
                SSIT CAMPUS ASSISTANT
            </div>

            <div class="subtitle">
                OFFICIAL STUDENT INFORMATION PORTAL
            </div>

            <div class="tagline">
                Academic Guidance · Campus Services · Student Support
            </div>
        """)

    gr.Markdown(
        "## Welcome to the Campus Help Desk",
        elem_id="welcome"
    )

    gr.Markdown(
        "Ask questions about college academics, attendance, "
        "examinations, departments, library and hostel facilities."
    )

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(height=420, elem_id="chatbot"),
        textbox=gr.Textbox(
            placeholder="Type your campus question here...",
            lines=2
        ),
        examples=[
            "How can I get information about attendance?",
            "Where can I find the examination timetable?",
            "Tell me about hostel facilities.",
            "How do I get admission information?",
            "Where can I find library information?"
        ],
        submit_btn="Ask Assistant"
    )

    gr.HTML("""
        <div style="
            text-align:center;
            padding:20px 10px;
            color:#414A29;
            font-size:13px;
        ">
            SSIT Campus Assistant · Student Information Portal
            <br>
            Please verify important information through official
            college sources.
        </div>
    """)


# ==============================================
# 5. LAUNCH THE APPLICATION
# ==============================================

demo.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="lime",
        secondary_hue="amber",
        neutral_hue="stone"
    ),
    css=custom_css
)


Upload your SSIT college logo (PNG or JPG).


Saving ssit_logo.png.jpeg to ssit_logo.png.jpeg
Logo loaded: ssit_logo.png.jpeg


/tmp/ipykernel_2295/3836522289.py:264: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://56483e5dd46b3c0fe3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:

# =====================================================
# SSIT CAMPUS ASSISTANT — COMPLETE GOOGLE COLAB CODE
# Logo: Top-left corner | Title: Centered
# =====================================================

!pip install -q -U gradio

import os
import gradio as gr
from google.colab import files

# =====================================================
# 1. UPLOAD YOUR SSIT LOGO
# =====================================================

print("Upload your SSIT logo image (.png, .jpg, or .jpeg)")
uploaded = files.upload()

LOGO_PATH = None

for filename in uploaded.keys():
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        LOGO_PATH = os.path.abspath(filename)
        break

if LOGO_PATH is None:
    raise ValueError("Please upload a valid PNG, JPG, JPEG, or WEBP image.")

print("Logo loaded successfully:", os.path.basename(LOGO_PATH))


# =====================================================
# 2. CSS — OLIVE GREEN AND GOLD THEME
# =====================================================

custom_css = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
    background: #F5F2E8 !important;
    font-family: Arial, sans-serif !important;
}

body {
    background: #F5F2E8 !important;
}

/* Main college header */
#hero {
    background: linear-gradient(135deg, #252D18, #414A29);
    color: #F5F2E8;
    padding: 32px 24px;
    border: 2px solid #B89A4A;
    border-radius: 18px;
    position: relative;
    text-align: center;
    margin: 12px 0 28px 0;
    min-height: 180px;
    box-shadow: 0 5px 16px rgba(0,0,0,0.12);
}

/* Logo in top-left corner */
#college-logo {
    position: absolute !important;
    top: 20px !important;
    left: 20px !important;
    width: 100px !important;
    height: 100px !important;
    min-width: 100px !important;
    margin: 0 !important;
    padding: 0 !important;
    background: white !important;
    border: 2px solid #D6B65B !important;
    border-radius: 10px !important;
    overflow: hidden !important;
    z-index: 5 !important;
}

#college-logo img {
    width: 100% !important;
    height: 100% !important;
    object-fit: contain !important;
}

/* Centered header text */
.college-name {
    color: #F5F2E8 !important;
    font-size: 19px !important;
    font-weight: 600 !important;
    text-align: center !important;
    letter-spacing: 0.5px;
    padding: 8px 110px 0 110px;
}

.gold-line {
    width: 220px;
    height: 3px;
    background: #D6B65B;
    margin: 18px auto;
    border-radius: 5px;
}

.main-title {
    color: #D6B65B !important;
    font-size: 36px !important;
    font-weight: 800 !important;
    text-align: center !important;
    margin: 12px 0 !important;
}

.subtitle {
    color: #FFFFFF !important;
    font-size: 18px !important;
    text-align: center !important;
    margin: 12px 0 !important;
}

.tagline {
    color: #F5F2E8 !important;
    font-size: 16px !important;
    text-align: center !important;
}

/* Chat section */
#welcome {
    color: #252D18 !important;
}

#chatbot {
    border: 1px solid #B89A4A !important;
    border-radius: 16px !important;
    background: #FFFCF5 !important;
}

button.primary {
    background: #414A29 !important;
    color: white !important;
    border: 1px solid #B89A4A !important;
}

button.primary:hover {
    background: #252D18 !important;
}

footer {
    display: none !important;
}

/* Mobile layout */
@media (max-width: 600px) {
    #hero {
        padding: 100px 12px 24px 12px;
    }

    #college-logo {
        width: 65px !important;
        height: 65px !important;
        min-width: 65px !important;
        top: 12px !important;
        left: 12px !important;
    }

    .college-name {
        font-size: 14px !important;
        padding: 8px 0 !important;
    }

    .main-title {
        font-size: 26px !important;
    }

    .subtitle {
        font-size: 15px !important;
    }

    .tagline {
        font-size: 13px !important;
    }
}
"""


# =====================================================
# 3. CHATBOT FUNCTION
# =====================================================

def chatbot(message, history):
    question = message.strip()

    if not question:
        return "Please enter your campus-related question."

    q = question.lower()

    if any(word in q for word in ["hello", "hi", "hey"]):
        return (
            "Hello! Welcome to the SSIT Campus Assistant. "
            "How can I help you today?"
        )

    if "attendance" in q:
        return (
            "For attendance rules and your current attendance, "
            "please check the official college portal or contact "
            "your department office."
        )

    if any(word in q for word in ["exam", "examination", "timetable"]):
        return (
            "For the latest examination timetable, please check "
            "official SSIT notices or contact your department."
        )

    if any(word in q for word in ["hostel", "mess", "accommodation"]):
        return (
            "For hostel accommodation, mess timings and facilities, "
            "please contact the SSIT hostel office."
        )

    if any(word in q for word in ["department", "course", "branch"]):
        return (
            "Please mention the department or course you want "
            "information about. Connect verified college documents "
            "for detailed answers."
        )

    if any(word in q for word in ["admission", "fee", "fees"]):
        return (
            "For current admission requirements and fee details, "
            "please consult the official SSIT admissions office."
        )

    if any(word in q for word in ["library", "books"]):
        return (
            "For library timings, book availability and library rules, "
            "please check official SSIT library information."
        )

    return (
        "Welcome to the SSIT Campus Assistant! This is a demo response. "
        "Connect your RAG pipeline and college documents to answer "
        "questions using verified SSIT information."
    )


# =====================================================
# 4. BUILD THE CHATBOT INTERFACE
# =====================================================

with gr.Blocks(title="SSIT | Campus Assistant") as demo:

    # Header with logo positioned at the left
    with gr.Column(elem_id="hero"):

        gr.Image(
            value=LOGO_PATH,
            show_label=False,
            container=False,
            interactive=False,
            height=100,
            width=100,
            elem_id="college-logo"
        )

        gr.HTML("""
            <div class="college-name">
                SRI SIDDHARTHA INSTITUTE OF TECHNOLOGY
            </div>

            <div class="gold-line"></div>

            <div class="main-title">
                SSIT CAMPUS ASSISTANT
            </div>

            <div class="subtitle">
                OFFICIAL STUDENT INFORMATION PORTAL
            </div>

            <div class="tagline">
                Academic Guidance · Campus Services · Student Support
            </div>
        """)

    # Welcome section
    gr.Markdown(
        "## Welcome to the Campus Help Desk",
        elem_id="welcome"
    )

    gr.Markdown(
        "Ask questions about college academics, attendance, "
        "examinations, departments, library and hostel facilities."
    )

    # Chat interface
    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(height=420, elem_id="chatbot"),
        textbox=gr.Textbox(
            placeholder="Type your campus question here...",
            lines=2
        ),
        examples=[
            "How can I get information about attendance?",
            "Where can I find the examination timetable?",
            "Tell me about hostel facilities.",
            "How do I get admission information?",
            "Where can I find library information?"
        ],
        submit_btn="Ask Assistant"
    )

    # Footer
    gr.HTML("""
        <div style="
            text-align:center;
            padding:20px 10px;
            color:#414A29;
            font-size:13px;
        ">
            SSIT Campus Assistant · Student Information Portal
            <br>
            Please verify important information through official
            college sources.
        </div>
    """)


# =====================================================
# 5. LAUNCH
# =====================================================

demo.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="lime",
        secondary_hue="amber",
        neutral_hue="stone"
    ),
    css=custom_css
)


Upload your SSIT logo image (.png, .jpg, or .jpeg)


Saving ssit_logo.png.jpeg to ssit_logo.png (1).jpeg
Logo loaded successfully: ssit_logo.png (1).jpeg


/tmp/ipykernel_2295/3671114286.py:297: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://794e35a07fcedab6b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [34]:

!pip install -q -U gradio

import os
import gradio as gr
from google.colab import files

# ==========================================
# 1. UPLOAD BOTH IMAGES
# ==========================================
print("Upload your SSIT LOGO image:")
logo_upload = files.upload()

LOGO_PATH = None
for filename in logo_upload.keys():
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        LOGO_PATH = os.path.abspath(filename)
        break

if LOGO_PATH is None:
    raise ValueError("Please upload a valid SSIT logo image.")

print("\nNow upload your GLOBE BACKGROUND image:")
globe_upload = files.upload()

GLOBE_PATH = None
for filename in globe_upload.keys():
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        GLOBE_PATH = os.path.abspath(filename)
        break

if GLOBE_PATH is None:
    raise ValueError("Please upload a valid globe image.")

print("\nBoth images uploaded successfully!")


# ==========================================
# 2. BACKGROUND IMAGE CSS
# ==========================================
custom_css = """
/* Main page background */
body, .gradio-container {
    background:
        linear-gradient(rgba(12, 17, 13, 0.70),
                        rgba(12, 17, 13, 0.78)),
        url("file/GLOBE_IMAGE_PLACEHOLDER") center center / cover no-repeat fixed !important;
    color: #F5F1E5 !important;
}

/* Main application layout */
.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    font-family: Arial, sans-serif !important;
    padding: 20px !important;
}

/* Header */
#hero {
    position: relative !important;
    background: rgba(22, 29, 20, 0.68) !important;
    border: 2px solid #B89A4A !important;
    border-radius: 18px !important;
    padding: 30px 20px 25px !important;
    min-height: 185px !important;
    text-align: center !important;
    backdrop-filter: blur(3px);
    margin-bottom: 25px !important;
}

/* Logo at top left */
#college-logo {
    position: absolute !important;
    top: 18px !important;
    left: 18px !important;
    width: 90px !important;
    height: 90px !important;
    min-width: 90px !important;
    margin: 0 !important;
    padding: 4px !important;
    background: white !important;
    border: 2px solid #D6B65B !important;
    border-radius: 10px !important;
    overflow: hidden !important;
    z-index: 10 !important;
}

#college-logo img {
    width: 100% !important;
    height: 100% !important;
    object-fit: contain !important;
}

/* Center heading */
.college-name {
    color: #E8D59A !important;
    font-size: 19px !important;
    font-weight: bold !important;
    letter-spacing: 1px !important;
    padding: 4px 100px 8px !important;
}

.gold-line {
    width: 120px;
    height: 3px;
    background: #D6B65B;
    margin: 8px auto 12px;
    border-radius: 5px;
}

.main-title {
    color: #FFFFFF !important;
    font-size: 32px !important;
    font-weight: 800 !important;
    letter-spacing: 2px !important;
    margin: 5px 0 !important;
    text-shadow: 0 2px 8px #000000;
}

.subtitle {
    color: #E8D59A !important;
    font-size: 13px !important;
    font-weight: bold !important;
    letter-spacing: 2px !important;
    margin-top: 8px !important;
}

.tagline {
    color: #F4F0E5 !important;
    font-size: 13px !important;
    margin-top: 12px !important;
}

/* Welcome section */
#welcome {
    color: #F3D98B !important;
}

/* Chat panel */
#chatbot {
    border: 1px solid #B89A4A !important;
    border-radius: 14px !important;
    background: rgba(20, 25, 20, 0.82) !important;
}

/* General panels */
.gr-panel, .gr-box, .gr-group {
    background: rgba(24, 30, 23, 0.82) !important;
    border-color: rgba(214, 182, 91, 0.55) !important;
}

/* Inputs */
textarea, input {
    background: rgba(250, 248, 239, 0.97) !important;
    color: #20251C !important;
    border: 1px solid #B89A4A !important;
    border-radius: 10px !important;
}

/* Buttons */
button {
    border-radius: 9px !important;
}

/* Mobile layout */
@media (max-width: 650px) {
    .gradio-container {
        padding: 8px !important;
    }

    #hero {
        padding: 115px 12px 20px !important;
    }

    #college-logo {
        top: 12px !important;
        left: calc(50% - 40px) !important;
        width: 80px !important;
        height: 80px !important;
        min-width: 80px !important;
    }

    .college-name {
        padding: 0 !important;
        font-size: 14px !important;
    }

    .main-title {
        font-size: 23px !important;
        letter-spacing: 1px !important;
    }

    .subtitle {
        font-size: 11px !important;
        letter-spacing: 1px !important;
    }
}
"""

# Replace placeholder with the uploaded background image path.
# Gradio serves files from the working directory using /gradio_api/file=...
# Use the supported static file route in the CSS.
GLOBE_URL = "/gradio_api/file=" + GLOBE_PATH.replace("/content/", "")
custom_css = custom_css.replace(
    "file/GLOBE_IMAGE_PLACEHOLDER",
    GLOBE_URL
)


# ==========================================
# 3. SAMPLE CHATBOT
# ==========================================
def chatbot(message, history):
    question = message.lower().strip()

    if any(word in question for word in ["hello", "hi", "hey"]):
        return "Hello! Welcome to the SSIT Campus Assistant. How can I help you?"

    elif "attendance" in question:
        return (
            "Attendance requirements can vary by programme and current "
            "college rules. Please check the latest official SSIT regulations "
            "or contact your department office."
        )

    elif any(word in question for word in ["exam", "timetable", "schedule"]):
        return (
            "For examination timetables and updates, please check the official "
            "college notice board or contact your department."
        )

    elif any(word in question for word in ["hostel", "mess"]):
        return (
            "For hostel availability, mess timings and current rules, "
            "please contact the SSIT hostel office."
        )

    elif any(word in question for word in ["department", "course", "branch"]):
        return (
            "Please mention the department or course you want to know about. "
            "For official details, check the SSIT website or department office."
        )

    elif any(word in question for word in ["admission", "fees"]):
        return (
            "For current admission requirements and fee details, please "
            "confirm with the SSIT admissions office."
        )

    elif "library" in question:
        return (
            "For library hours, book availability and borrowing rules, "
            "please check with the SSIT library."
        )

    return (
        "Thanks for your question! This is the interface demo. Connect your "
        "SSIT documents and RAG pipeline to answer questions using verified "
        "college information."
    )


# ==========================================
# 4. BUILD THE INTERFACE
# ==========================================
with gr.Blocks(title="SSIT | Campus Assistant") as demo:

    with gr.Column(elem_id="hero"):
        gr.Image(
            value=LOGO_PATH,
            show_label=False,
            container=False,
            interactive=False,
            height=90,
            width=90,
            elem_id="college-logo"
        )

        gr.HTML("""
            <div class="college-name">
                SRI SIDDHARTHA INSTITUTE OF TECHNOLOGY
            </div>
            <div class="gold-line"></div>
            <div class="main-title">SSIT CAMPUS ASSISTANT</div>
            <div class="subtitle">
                OFFICIAL STUDENT INFORMATION PORTAL
            </div>
            <div class="tagline">
                Academic Guidance · Campus Services · Student Support
            </div>
        """)

    gr.Markdown(
        "## Welcome to the Campus Help Desk",
        elem_id="welcome"
    )

    gr.Markdown(
        "Ask about academics, attendance, examinations, departments, "
        "library services and hostel facilities."
    )

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(height=420, elem_id="chatbot"),
        textbox=gr.Textbox(
            placeholder="Type your campus question here...",
            lines=2
        ),
        examples=[
            "What are the attendance requirements?",
            "Where can I find the exam timetable?",
            "Tell me about hostel facilities",
            "How can I get admission information?"
        ],
        submit_btn="Ask Assistant"
    )


# ==========================================
# 5. LAUNCH
# ==========================================
demo.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="lime",
        secondary_hue="amber",
        neutral_hue="stone"
    ),
    css=custom_css
)


Upload your SSIT LOGO image:


Saving ssit_logo.png.jpeg to ssit_logo.png (2).jpeg

Now upload your GLOBE BACKGROUND image:


Saving image_522376 - Copy.jpg to image_522376 - Copy.jpg

Both images uploaded successfully!


/tmp/ipykernel_2295/2272349904.py:303: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71e9cca68d8272fb5a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [35]:

!pip install -q -U gradio

import os
import gradio as gr
from google.colab import files
from urllib.parse import quote

# ==================================================
# 1. UPLOAD THE SSIT LOGO
# ==================================================
print("Upload your SSIT college logo:")
logo_upload = files.upload()

LOGO_PATH = None

for filename in logo_upload:
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        LOGO_PATH = os.path.abspath(filename)
        break

if not LOGO_PATH:
    raise ValueError("Please upload a valid SSIT logo image.")

print("Logo uploaded:", os.path.basename(LOGO_PATH))


# ==================================================
# 2. UPLOAD THE GLOBE BACKGROUND IMAGE
# ==================================================
print("\nNow upload your globe background image:")
globe_upload = files.upload()

GLOBE_PATH = None

for filename in globe_upload:
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        GLOBE_PATH = os.path.abspath(filename)
        break

if not GLOBE_PATH:
    raise ValueError("Please upload a valid globe image.")

print("Globe uploaded:", os.path.basename(GLOBE_PATH))


# ==================================================
# 3. CREATE A SERVED URL FOR THE GLOBE IMAGE
# ==================================================
GLOBE_URL = "/gradio_api/file=" + quote(GLOBE_PATH, safe="/")


# ==================================================
# 4. CUSTOM CSS
# ==================================================
custom_css = """
/* Full-page globe background */
html, body {
    min-height: 100% !important;
    background: #101710 !important;
}

.gradio-container {
    max-width: 1250px !important;
    min-height: 100vh !important;
    margin: auto !important;
    padding: 20px !important;
    font-family: Arial, sans-serif !important;
    color: #F5F1E5 !important;

    background-image:
        linear-gradient(
            rgba(8, 13, 10, 0.48),
            rgba(8, 13, 10, 0.62)
        ),
        url("GLOBE_IMAGE_PLACEHOLDER") !important;

    background-size: cover !important;
    background-position: center center !important;
    background-repeat: no-repeat !important;
    background-attachment: fixed !important;
}

/* Header panel */
#hero {
    position: relative !important;
    box-sizing: border-box !important;
    min-height: 220px !important;
    padding: 30px 20px 25px !important;
    margin-bottom: 25px !important;

    background: rgba(22, 29, 20, 0.68) !important;
    border: 2px solid #B89A4A !important;
    border-radius: 18px !important;
    text-align: center !important;
    backdrop-filter: blur(3px);
}

/* College logo: upper-left corner */
#college-logo {
    position: absolute !important;
    top: 18px !important;
    left: 18px !important;
    width: 95px !important;
    height: 95px !important;
    min-width: 95px !important;
    margin: 0 !important;
    padding: 4px !important;
    box-sizing: border-box !important;

    background: #FFFFFF !important;
    border: 2px solid #D6B65B !important;
    border-radius: 12px !important;
    overflow: hidden !important;
    z-index: 10 !important;
}

/* Keep the logo inside its frame */
#college-logo img {
    width: 100% !important;
    height: 100% !important;
    object-fit: contain !important;
}

/* College name */
.college-name {
    color: #E8D59A !important;
    font-size: 19px !important;
    font-weight: 700 !important;
    letter-spacing: 1px !important;
    line-height: 1.5 !important;
    padding: 5px 100px 8px !important;
}

/* Gold divider */
.gold-line {
    width: 125px;
    height: 3px;
    background: #D6B65B;
    margin: 8px auto 14px;
    border-radius: 5px;
}

/* Main title */
.main-title {
    color: #FFFFFF !important;
    font-size: 34px !important;
    font-weight: 800 !important;
    letter-spacing: 2px !important;
    line-height: 1.25 !important;
    margin: 5px 0 10px !important;
    text-shadow: 0 2px 8px #000000;
}

/* Subtitle */
.subtitle {
    color: #E8D59A !important;
    font-size: 13px !important;
    font-weight: bold !important;
    letter-spacing: 2px !important;
    margin-top: 8px !important;
}

/* Tagline */
.tagline {
    color: #FFFFFF !important;
    font-size: 14px !important;
    margin-top: 12px !important;
}

/* Welcome heading and description */
#welcome {
    color: #F3D98B !important;
}

.gradio-container p,
.gradio-container h2 {
    color: #FFFFFF !important;
}

/* Chat panel */
#chatbot {
    border: 1px solid #B89A4A !important;
    border-radius: 15px !important;
    background: rgba(18, 24, 19, 0.84) !important;
}

/* Chat messages and panels */
.gr-panel,
.gr-box,
.gr-group {
    background-color: rgba(20, 27, 21, 0.85) !important;
    border-color: rgba(214, 182, 91, 0.60) !important;
}

/* Input boxes */
textarea,
input {
    background: #FAF8EF !important;
    color: #20251C !important;
    border: 1px solid #B89A4A !important;
    border-radius: 10px !important;
}

/* Buttons */
button {
    border-radius: 9px !important;
}

/* Mobile screens */
@media (max-width: 650px) {
    .gradio-container {
        padding: 8px !important;
        background-attachment: scroll !important;
    }

    #hero {
        padding: 125px 12px 22px !important;
        min-height: 260px !important;
    }

    #college-logo {
        top: 12px !important;
        left: calc(50% - 40px) !important;
        width: 80px !important;
        height: 80px !important;
        min-width: 80px !important;
    }

    .college-name {
        padding: 0 !important;
        font-size: 14px !important;
    }

    .main-title {
        font-size: 24px !important;
        letter-spacing: 1px !important;
    }

    .subtitle {
        font-size: 11px !important;
        letter-spacing: 1px !important;
    }

    .tagline {
        font-size: 12px !important;
    }
}
"""

# Insert the real image URL into the CSS
custom_css = custom_css.replace(
    "GLOBE_IMAGE_PLACEHOLDER",
    GLOBE_URL
)


# ==================================================
# 5. SAMPLE CHATBOT FUNCTION
# ==================================================
def chatbot(message, history):
    question = message.lower().strip()

    if any(word in question for word in ["hello", "hi", "hey"]):
        return (
            "Hello! Welcome to the SSIT Campus Assistant. "
            "How can I help you today?"
        )

    elif "attendance" in question:
        return (
            "For attendance requirements, please check the latest "
            "official SSIT regulations or contact your department office."
        )

    elif any(word in question for word in ["exam", "timetable", "schedule"]):
        return (
            "For examination timetables and updates, please check "
            "the official college notice board or your department."
        )

    elif any(word in question for word in ["hostel", "mess"]):
        return (
            "For hostel availability, mess timings and current rules, "
            "please contact the SSIT hostel office."
        )

    elif any(word in question for word in ["department", "course", "branch"]):
        return (
            "Please mention which department or course you want to know "
            "about. Confirm official details with the relevant department."
        )

    elif any(word in question for word in ["admission", "fees"]):
        return (
            "For current admission requirements and fees, please contact "
            "the SSIT admissions office."
        )

    elif "library" in question:
        return (
            "For library hours, book availability and borrowing rules, "
            "please contact the SSIT library."
        )

    else:
        return (
            "Thanks for your question! This is the interface demo. "
            "Connect your SSIT documents and RAG pipeline to provide "
            "answers based on official college information."
        )


# ==================================================
# 6. BUILD THE SSIT CAMPUS ASSISTANT
# ==================================================
with gr.Blocks(title="SSIT | Campus Assistant") as demo:

    with gr.Column(elem_id="hero"):

        gr.Image(
            value=LOGO_PATH,
            show_label=False,
            container=False,
            interactive=False,
            height=95,
            width=95,
            elem_id="college-logo"
        )

        gr.HTML("""
            <div class="college-name">
                SRI SIDDHARTHA INSTITUTE OF TECHNOLOGY
            </div>

            <div class="gold-line"></div>

            <div class="main-title">
                SSIT CAMPUS ASSISTANT
            </div>

            <div class="subtitle">
                OFFICIAL STUDENT INFORMATION PORTAL
            </div>

            <div class="tagline">
                Academic Guidance · Campus Services · Student Support
            </div>
        """)

    gr.Markdown(
        "## Welcome to the Campus Help Desk",
        elem_id="welcome"
    )

    gr.Markdown(
        "Ask about academics, attendance, examinations, departments, "
        "library services and hostel facilities."
    )

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(height=420, elem_id="chatbot"),
        textbox=gr.Textbox(
            placeholder="Type your campus question here...",
            lines=2
        ),
        examples=[
            "What are the attendance requirements?",
            "Where can I find the exam timetable?",
            "Tell me about hostel facilities",
            "How can I get admission information?"
        ],
        submit_btn="Ask Assistant"
    )


# ==================================================
# 7. LAUNCH THE WEBSITE
# ==================================================
demo.launch(
    share=True,
    allowed_paths=[LOGO_PATH, GLOBE_PATH],
    theme=gr.themes.Soft(
        primary_hue="lime",
        secondary_hue="amber",
        neutral_hue="stone"
    ),
    css=custom_css
)


Upload your SSIT college logo:


Saving ssit_logo.png.jpeg to ssit_logo.png (3).jpeg
Logo uploaded: ssit_logo.png (3).jpeg

Now upload your globe background image:


Saving image_522376 - Copy.jpg to image_522376 - Copy (1).jpg
Globe uploaded: image_522376 - Copy (1).jpg


/tmp/ipykernel_2295/1933931158.py:360: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://657a0a600fb55e5939.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [36]:

!pip install -q -U gradio

import os
import gradio as gr
from google.colab import files
from urllib.parse import quote

# ==========================================
# 1. UPLOAD SSIT LOGO
# ==========================================
print("Upload your SSIT logo:")
logo_upload = files.upload()

LOGO_PATH = next(
    (os.path.abspath(f) for f in logo_upload
     if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))),
    None
)

if not LOGO_PATH:
    raise ValueError("Please upload a valid logo image.")


# ==========================================
# 2. UPLOAD GLOBE BACKGROUND
# ==========================================
print("Upload your globe background image:")
globe_upload = files.upload()

GLOBE_PATH = next(
    (os.path.abspath(f) for f in globe_upload
     if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))),
    None
)

if not GLOBE_PATH:
    raise ValueError("Please upload a valid globe image.")

GLOBE_URL = "/gradio_api/file=" + quote(GLOBE_PATH, safe="/")


# ==========================================
# 3. CUSTOM CSS
# ==========================================
custom_css = """
html, body {
    min-height: 100%;
    background: #101710 !important;
}

.gradio-container {
    max-width: 1250px !important;
    min-height: 100vh !important;
    margin: auto !important;
    padding: 20px !important;
    font-family: Arial, sans-serif !important;
    color: #FFFFFF !important;

    background-image:
        linear-gradient(
            rgba(8, 13, 10, 0.40),
            rgba(8, 13, 10, 0.55)
        ),
        url("GLOBE_IMAGE_PLACEHOLDER") !important;

    background-size: cover !important;
    background-position: center center !important;
    background-repeat: no-repeat !important;
    background-attachment: fixed !important;
}

/* HEADER */
#hero {
    position: relative !important;
    box-sizing: border-box !important;
    min-height: 220px !important;
    padding: 30px 20px 25px !important;
    margin-bottom: 25px !important;
    background: rgba(22, 29, 20, 0.75) !important;
    border: 2px solid #B89A4A !important;
    border-radius: 18px !important;
    text-align: center !important;
}

/* LOGO TOP LEFT */
#college-logo {
    position: absolute !important;
    top: 18px !important;
    left: 18px !important;
    width: 95px !important;
    height: 95px !important;
    min-width: 95px !important;
    margin: 0 !important;
    padding: 4px !important;
    background: white !important;
    border: 2px solid #D6B65B !important;
    border-radius: 12px !important;
    overflow: hidden !important;
    z-index: 10 !important;
}

#college-logo img {
    width: 100% !important;
    height: 100% !important;
    object-fit: contain !important;
}

/* HEADER TEXT */
.college-name {
    color: #E8D59A !important;
    font-size: 19px !important;
    font-weight: bold !important;
    letter-spacing: 1px !important;
    padding: 5px 100px 8px !important;
}

.gold-line {
    width: 125px;
    height: 3px;
    background: #D6B65B;
    margin: 8px auto 14px;
    border-radius: 5px;
}

.main-title {
    color: white !important;
    font-size: 34px !important;
    font-weight: 800 !important;
    letter-spacing: 2px !important;
    margin: 5px 0 10px !important;
    text-shadow: 0 2px 8px black;
}

.subtitle {
    color: #E8D59A !important;
    font-size: 13px !important;
    font-weight: bold !important;
    letter-spacing: 2px !important;
}

.tagline {
    color: white !important;
    font-size: 14px !important;
    margin-top: 12px !important;
}

/* WELCOME TEXT */
#welcome {
    color: #FFE29A !important;
}

/* MAKE DESCRIPTION TEXT WHITE */
.gradio-container p {
    color: white !important;
}

/* CHAT PANEL */
#chatbot {
    border: 1px solid #B89A4A !important;
    border-radius: 15px !important;
    background: rgba(18, 24, 19, 0.82) !important;
}

/* ==========================================
   SUGGESTED QUESTION BUTTONS - BLACK TEXT
   ========================================== */

/* Gradio example question buttons */
.gradio-container button[class*="example"],
.gradio-container .examples button,
.gradio-container button[data-testid*="example"],
.gradio-container [class*="example"] button {
    background: #FFFDF4 !important;
    color: #111111 !important;
    border: 1px solid #D6B65B !important;
    border-radius: 12px !important;
}

/* Force all text inside suggestion buttons to black */
.gradio-container button[class*="example"] *,
.gradio-container .examples button *,
.gradio-container button[data-testid*="example"] *,
.gradio-container [class*="example"] button * {
    color: #111111 !important;
    opacity: 1 !important;
}

/* Hover effect */
.gradio-container button[class*="example"]:hover,
.gradio-container .examples button:hover,
.gradio-container button[data-testid*="example"]:hover,
.gradio-container [class*="example"] button:hover {
    background: #F1E4B8 !important;
    border-color: #B89A4A !important;
}

/* Chat messages/panels */
.gr-panel,
.gr-box,
.gr-group {
    border-color: rgba(214, 182, 91, 0.60) !important;
}

/* Text input */
textarea, input {
    background: #FFFDF4 !important;
    color: #111111 !important;
    border: 1px solid #B89A4A !important;
    border-radius: 10px !important;
}

/* Other buttons */
button {
    border-radius: 9px !important;
}

/* MOBILE */
@media (max-width: 650px) {
    .gradio-container {
        padding: 8px !important;
        background-attachment: scroll !important;
    }

    #hero {
        padding: 125px 12px 22px !important;
        min-height: 260px !important;
    }

    #college-logo {
        top: 12px !important;
        left: calc(50% - 40px) !important;
        width: 80px !important;
        height: 80px !important;
        min-width: 80px !important;
    }

    .college-name {
        padding: 0 !important;
        font-size: 14px !important;
    }

    .main-title {
        font-size: 24px !important;
        letter-spacing: 1px !important;
    }

    .subtitle {
        font-size: 11px !important;
        letter-spacing: 1px !important;
    }

    .tagline {
        font-size: 12px !important;
    }
}
"""

custom_css = custom_css.replace(
    "GLOBE_IMAGE_PLACEHOLDER",
    GLOBE_URL
)


# ==========================================
# 4. CHATBOT DEMO
# ==========================================
def chatbot(message, history):
    q = message.lower().strip()

    if any(x in q for x in ["hello", "hi", "hey"]):
        return "Hello! Welcome to the SSIT Campus Assistant. How can I help?"

    elif "attendance" in q:
        return (
            "For attendance requirements, check the latest official "
            "SSIT regulations or contact your department office."
        )

    elif any(x in q for x in ["exam", "timetable", "schedule"]):
        return (
            "For examination timetables and updates, check the official "
            "college notice board or your department."
        )

    elif any(x in q for x in ["hostel", "mess"]):
        return (
            "For hostel availability, mess timings and current rules, "
            "please contact the SSIT hostel office."
        )

    elif any(x in q for x in ["department", "course", "branch"]):
        return (
            "Please specify the department or course you want to know about. "
            "Confirm official details with the relevant department."
        )

    elif any(x in q for x in ["admission", "fees"]):
        return (
            "For current admission requirements and fees, contact "
            "the SSIT admissions office."
        )

    elif "library" in q:
        return (
            "For library hours, book availability and borrowing rules, "
            "please contact the SSIT library."
        )

    return (
        "This is the interface demo. Connect your SSIT documents and RAG "
        "pipeline to provide answers based on official college information."
    )


# ==========================================
# 5. BUILD INTERFACE
# ==========================================
with gr.Blocks(title="SSIT | Campus Assistant") as demo:

    with gr.Column(elem_id="hero"):
        gr.Image(
            value=LOGO_PATH,
            show_label=False,
            container=False,
            interactive=False,
            height=95,
            width=95,
            elem_id="college-logo"
        )

        gr.HTML("""
            <div class="college-name">
                SRI SIDDHARTHA INSTITUTE OF TECHNOLOGY
            </div>
            <div class="gold-line"></div>
            <div class="main-title">SSIT CAMPUS ASSISTANT</div>
            <div class="subtitle">
                OFFICIAL STUDENT INFORMATION PORTAL
            </div>
            <div class="tagline">
                Academic Guidance · Campus Services · Student Support
            </div>
        """)

    gr.Markdown(
        "## Welcome to the Campus Help Desk",
        elem_id="welcome"
    )

    gr.Markdown(
        "Ask about academics, attendance, examinations, departments, "
        "library services and hostel facilities."
    )

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(height=420, elem_id="chatbot"),
        textbox=gr.Textbox(
            placeholder="Type your campus question here...",
            lines=2
        ),
        examples=[
            "What are the attendance requirements?",
            "Where can I find the exam timetable?",
            "Tell me about hostel facilities",
            "How can I get admission information?"
        ],
        submit_btn="Ask Assistant"
    )


# ==========================================
# 6. LAUNCH
# ==========================================
demo.launch(
    share=True,
    allowed_paths=[LOGO_PATH, GLOBE_PATH],
    theme=gr.themes.Soft(
        primary_hue="lime",
        secondary_hue="amber",
        neutral_hue="stone"
    ),
    css=custom_css
)


Upload your SSIT logo:


Saving ssit_logo.png.jpeg to ssit_logo.png (4).jpeg
Upload your globe background image:


Saving image_522376 - Copy.jpg to image_522376 - Copy (2).jpg


/tmp/ipykernel_2295/370830157.py:355: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d72f1161962d31ba70.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [37]:

!pip install -q -U gradio

import os
import gradio as gr
from google.colab import files
from urllib.parse import quote

# ==========================================
# 1. UPLOAD SSIT LOGO
# ==========================================
print("Upload your SSIT logo:")
logo_upload = files.upload()

LOGO_PATH = next(
    (os.path.abspath(f) for f in logo_upload
     if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))),
    None
)

if not LOGO_PATH:
    raise ValueError("Please upload a valid SSIT logo image.")


# ==========================================
# 2. UPLOAD GLOBE BACKGROUND
# ==========================================
print("Upload your globe background image:")
globe_upload = files.upload()

GLOBE_PATH = next(
    (os.path.abspath(f) for f in globe_upload
     if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))),
    None
)

if not GLOBE_PATH:
    raise ValueError("Please upload a valid globe image.")

GLOBE_URL = "/gradio_api/file=" + quote(GLOBE_PATH, safe="/")


# ==========================================
# 3. CSS DESIGN
# ==========================================
custom_css = """
html, body {
    min-height: 100%;
    background: #101710 !important;
}

.gradio-container {
    max-width: 1250px !important;
    min-height: 100vh !important;
    margin: auto !important;
    padding: 20px !important;
    font-family: Arial, sans-serif !important;
    color: #FFFFFF !important;

    background-image:
        linear-gradient(
            rgba(8, 13, 10, 0.40),
            rgba(8, 13, 10, 0.55)
        ),
        url("GLOBE_IMAGE_PLACEHOLDER") !important;

    background-size: cover !important;
    background-position: center center !important;
    background-repeat: no-repeat !important;
    background-attachment: fixed !important;
}

/* HEADER */
#hero {
    position: relative !important;
    box-sizing: border-box !important;
    min-height: 220px !important;
    padding: 30px 20px 25px !important;
    margin-bottom: 25px !important;
    background: rgba(22, 29, 20, 0.75) !important;
    border: 2px solid #B89A4A !important;
    border-radius: 18px !important;
    text-align: center !important;
}

/* LOGO TOP LEFT */
#college-logo {
    position: absolute !important;
    top: 18px !important;
    left: 18px !important;
    width: 95px !important;
    height: 95px !important;
    min-width: 95px !important;
    margin: 0 !important;
    padding: 4px !important;
    background: white !important;
    border: 2px solid #D6B65B !important;
    border-radius: 12px !important;
    overflow: hidden !important;
    z-index: 10 !important;
}

#college-logo img {
    width: 100% !important;
    height: 100% !important;
    object-fit: contain !important;
}

/* HEADER TEXT */
.college-name {
    color: #E8D59A !important;
    font-size: 19px !important;
    font-weight: bold !important;
    letter-spacing: 1px !important;
    padding: 5px 100px 8px !important;
}

.gold-line {
    width: 125px;
    height: 3px;
    background: #D6B65B;
    margin: 8px auto 14px;
    border-radius: 5px;
}

.main-title {
    color: white !important;
    font-size: 34px !important;
    font-weight: 800 !important;
    letter-spacing: 2px !important;
    margin: 5px 0 10px !important;
    text-shadow: 0 2px 8px black;
}

.subtitle {
    color: #E8D59A !important;
    font-size: 13px !important;
    font-weight: bold !important;
    letter-spacing: 2px !important;
}

.tagline {
    color: white !important;
    font-size: 14px !important;
    margin-top: 12px !important;
}

/* WELCOME SECTION */
#welcome {
    color: #FFE29A !important;
}

.gradio-container p {
    color: white !important;
}

/* ==========================================
   SMALLER CHATBOT PANEL
   ========================================== */

/* Reduce overall chatbot height */
#chatbot {
    height: 300px !important;
    min-height: 300px !important;
    max-height: 300px !important;
    overflow-y: auto !important;

    border: 1px solid #B89A4A !important;
    border-radius: 15px !important;
    background: rgba(18, 24, 19, 0.82) !important;
}

/* Keep the chatbot interior compact */
#chatbot > div {
    min-height: 0 !important;
}

/* Compact suggested questions */
.gradio-container .examples {
    gap: 8px !important;
    margin-top: 8px !important;
}

/* Suggested question buttons: black text */
.gradio-container button[class*="example"],
.gradio-container .examples button,
.gradio-container button[data-testid*="example"],
.gradio-container [class*="example"] button {
    background: #FFFDF4 !important;
    color: #111111 !important;
    border: 1px solid #D6B65B !important;
    border-radius: 10px !important;
    min-height: 45px !important;
    padding: 10px 14px !important;
    font-size: 14px !important;
}

.gradio-container button[class*="example"] *,
.gradio-container .examples button *,
.gradio-container button[data-testid*="example"] *,
.gradio-container [class*="example"] button * {
    color: #111111 !important;
    opacity: 1 !important;
}

.gradio-container button[class*="example"]:hover,
.gradio-container .examples button:hover,
.gradio-container button[data-testid*="example"]:hover,
.gradio-container [class*="example"] button:hover {
    background: #F1E4B8 !important;
    border-color: #B89A4A !important;
}

/* Chat area and panels */
.gr-panel,
.gr-box,
.gr-group {
    border-color: rgba(214, 182, 91, 0.60) !important;
}

/* INPUT */
textarea, input {
    background: #FFFDF4 !important;
    color: #111111 !important;
    border: 1px solid #B89A4A !important;
    border-radius: 10px !important;
}

/* BUTTONS */
button {
    border-radius: 9px !important;
}

/* MOBILE */
@media (max-width: 650px) {
    .gradio-container {
        padding: 8px !important;
        background-attachment: scroll !important;
    }

    #hero {
        padding: 125px 12px 22px !important;
        min-height: 260px !important;
    }

    #college-logo {
        top: 12px !important;
        left: calc(50% - 40px) !important;
        width: 80px !important;
        height: 80px !important;
        min-width: 80px !important;
    }

    .college-name {
        padding: 0 !important;
        font-size: 14px !important;
    }

    .main-title {
        font-size: 24px !important;
        letter-spacing: 1px !important;
    }

    .subtitle {
        font-size: 11px !important;
        letter-spacing: 1px !important;
    }

    .tagline {
        font-size: 12px !important;
    }

    #chatbot {
        height: 260px !important;
        min-height: 260px !important;
        max-height: 260px !important;
    }

    .gradio-container button[class*="example"],
    .gradio-container .examples button,
    .gradio-container button[data-testid*="example"],
    .gradio-container [class*="example"] button {
        font-size: 12px !important;
        padding: 8px !important;
    }
}
"""

custom_css = custom_css.replace(
    "GLOBE_IMAGE_PLACEHOLDER",
    GLOBE_URL
)


# ==========================================
# 4. CHATBOT DEMO
# ==========================================
def chatbot(message, history):
    q = message.lower().strip()

    if any(x in q for x in ["hello", "hi", "hey"]):
        return "Hello! Welcome to the SSIT Campus Assistant. How can I help?"

    elif "attendance" in q:
        return (
            "For attendance requirements, check the latest official "
            "SSIT regulations or contact your department office."
        )

    elif any(x in q for x in ["exam", "timetable", "schedule"]):
        return (
            "For examination timetables and updates, check the official "
            "college notice board or your department."
        )

    elif any(x in q for x in ["hostel", "mess"]):
        return (
            "For hostel availability, mess timings and current rules, "
            "please contact the SSIT hostel office."
        )

    elif any(x in q for x in ["department", "course", "branch"]):
        return (
            "Please specify the department or course you want to know about. "
            "Confirm official details with the relevant department."
        )

    elif any(x in q for x in ["admission", "fees"]):
        return (
            "For current admission requirements and fees, contact "
            "the SSIT admissions office."
        )

    elif "library" in q:
        return (
            "For library hours, book availability and borrowing rules, "
            "please contact the SSIT library."
        )

    return (
        "This is the interface demo. Connect your SSIT documents and RAG "
        "pipeline to answer questions using official college information."
    )


# ==========================================
# 5. BUILD INTERFACE
# ==========================================
with gr.Blocks(title="SSIT | Campus Assistant") as demo:

    with gr.Column(elem_id="hero"):
        gr.Image(
            value=LOGO_PATH,
            show_label=False,
            container=False,
            interactive=False,
            height=95,
            width=95,
            elem_id="college-logo"
        )

        gr.HTML("""
            <div class="college-name">
                SRI SIDDHARTHA INSTITUTE OF TECHNOLOGY
            </div>
            <div class="gold-line"></div>
            <div class="main-title">SSIT CAMPUS ASSISTANT</div>
            <div class="subtitle">
                OFFICIAL STUDENT INFORMATION PORTAL
            </div>
            <div class="tagline">
                Academic Guidance · Campus Services · Student Support
            </div>
        """)

    gr.Markdown(
        "## Welcome to the Campus Help Desk",
        elem_id="welcome"
    )

    gr.Markdown(
        "Ask about academics, attendance, examinations, departments, "
        "library services and hostel facilities."
    )

    gr.ChatInterface(
        fn=chatbot,
        chatbot=gr.Chatbot(height=300, elem_id="chatbot"),
        textbox=gr.Textbox(
            placeholder="Type your campus question here...",
            lines=2
        ),
        examples=[
            "What are the attendance requirements?",
            "Where can I find the exam timetable?",
            "Tell me about hostel facilities",
            "How can I get admission information?"
        ],
        submit_btn="Ask Assistant"
    )


# ==========================================
# 6. LAUNCH
# ==========================================
demo.launch(
    share=True,
    allowed_paths=[LOGO_PATH, GLOBE_PATH],
    theme=gr.themes.Soft(
        primary_hue="lime",
        secondary_hue="amber",
        neutral_hue="stone"
    ),
    css=custom_css
)


Upload your SSIT logo:


Saving ssit_logo.png.jpeg to ssit_logo.png (5).jpeg
Upload your globe background image:


Saving image_522376 - Copy.jpg to image_522376 - Copy (3).jpg


/tmp/ipykernel_2295/2471963369.py:385: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://90ef178e8c49ed899a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
